# ROGII U-Continuity C1 Fork

Controlled fork of [`prvsiyan/rogii-public-score-frontier-lab-visuals`](https://www.kaggle.com/code/prvsiyan/rogii-public-score-frontier-lab-visuals). It first reconstructs and audits the exact public C0 candidate, then adds one target-free bounded C1 experiment.

The original C0 layer removes the value jump in `U = TVT + Z`. The added C1 layer reduces the local slope mismatch `dU/dMD` while preserving the first hidden point:

$$\Delta_{C1}(s)=a\,s\exp(-s/120),\qquad |\Delta_{C1}|\le 1\ \mathrm{ft}.$$

`a` is the clipped difference between exponentially weighted visible and hidden slopes measured over fixed 240 ft windows. Parameters are preregistered in code; no hidden labels or leaderboard feedback are used.

In [ ]:
from pathlib import Path
import glob, hashlib, json, os

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

BASE_SHA = "b192d3f348ae00680dc4df942b95cef5fd708c636a741f77dfb6b6e89b9ded4a"
FINAL_SHA = "50515fc1f872c9836e6d63a1990419ebf32316fe4521e654270b8e389a9199a3"
FINAL_IN_MEMORY_PRED_SHA = "7a0ab486440e47e131de7789e8e3377e61948c81e93b56e60d0e7c7d74ecd57c"
FINAL_RELOADED_PRED_SHA = "a7e6c03d0b5cddfd94c401ebc8cd272982ca581fd724d1458b5d39befb1b49ab"
CAP_FT = 8.0
TAU_MD_FT = 240.0
EXPECTED_ROWS = 14151
EXPECTED_WELLS = ("000d7d20", "00bbac68", "00e12e8b")
EXPECTED_IN_MEMORY_CHANGED_ROWS = 13993
EXPECTED_RELOADED_CHANGED_ROWS = 13927
WORK = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path(".")

plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams.update({
    "figure.figsize": (13, 7),
    "figure.dpi": 120,
    "savefig.dpi": 180,
    "axes.titleweight": "bold",
    "axes.spines.top": False,
    "axes.spines.right": False,
})

def file_sha(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

def prediction_sha(values):
    return hashlib.sha256(np.asarray(values, dtype="<f8").tobytes(order="C")).hexdigest()

def find_public_input(name):
    hits = sorted(Path(p) for p in glob.glob(f"/kaggle/input/**/{name}", recursive=True))
    exact = [p for p in hits if "rogii-f594-continuity8" in str(p)]
    local = os.environ.get("ROGII_PUBLIC_SOURCE_DIR")
    if not exact and local:
        candidate = Path(local) / name
        if candidate.exists():
            exact = [candidate]
    if len(exact) != 1:
        raise RuntimeError(f"expected one public continuity artifact {name}, found {exact or hits}")
    return exact[0]

def find_competition_root():
    roots = [
        Path("/kaggle/input/competitions/rogii-wellbore-geology-prediction"),
        Path("/kaggle/input/rogii-wellbore-geology-prediction"),
    ]
    local = os.environ.get("ROGII_COMPETITION_DATA_ROOT")
    if local:
        roots.append(Path(local))
    valid = [p for p in roots if (p / "test").exists() and (p / "sample_submission.csv").exists()]
    if len(valid) != 1:
        raise RuntimeError(f"expected one official competition root, found {valid}")
    return valid[0]

base_path = find_public_input("submission_before_continuity.csv")
public_final_path = find_public_input("submission.csv")
data_root = find_competition_root()
print("locked public control:", base_path)
print("public continuity cross-check:", public_final_path)
print("official competition root:", data_root)


## Deterministic reconstruction and strict submission contract

The next cell validates the public control, official sample order, hidden-suffix geometry, and visible/hidden boundary for every test well. It then applies only the preregistered cap-8/tau-240 fade and refuses to finish unless both the independently generated CSV and the public reference have the exact locked hash.


In [ ]:
base = pd.read_csv(base_path, dtype={"id": "string"})
if list(base.columns) != ["id", "tvt"]:
    raise RuntimeError(f"unexpected control columns: {list(base.columns)}")
if len(base) != EXPECTED_ROWS or not base["id"].is_unique:
    raise RuntimeError("unexpected control row count or duplicate ids")
if file_sha(base_path) != BASE_SHA:
    raise RuntimeError(f"public control hash changed: {file_sha(base_path)}")

sample = pd.read_csv(data_root / "sample_submission.csv", dtype={"id": "string"})
if list(sample.columns) != ["id", "tvt"] or not base["id"].equals(sample["id"]):
    raise RuntimeError("control does not match the official sample id order")

base_tvt = pd.to_numeric(base["tvt"], errors="coerce").to_numpy(dtype="float64")
if not np.isfinite(base_tvt).all():
    raise RuntimeError("control contains non-finite predictions")
final_tvt = base_tvt.copy()

parts = base["id"].astype(str).str.rsplit("_", n=1, expand=True)
if parts.shape[1] != 2:
    raise RuntimeError("expected ids formatted as well_rowindex")
work_rows = base.copy()
work_rows["well"] = parts[0].astype(str)
work_rows["row"] = pd.to_numeric(parts[1], errors="raise").astype(int)
if tuple(work_rows["well"].drop_duplicates()) != EXPECTED_WELLS:
    raise RuntimeError(f"unexpected test wells: {tuple(work_rows['well'].drop_duplicates())}")

report_rows = []
contexts = {}
for wid, group in work_rows.groupby("well", sort=False):
    hw_path = data_root / "test" / f"{wid}__horizontal_well.csv"
    if not hw_path.exists():
        raise FileNotFoundError(f"missing horizontal well: {hw_path}")
    hw = pd.read_csv(hw_path)
    required = {"MD", "Z", "TVT_input"}
    if not required.issubset(hw.columns):
        raise RuntimeError(f"well {wid} lacks columns {sorted(required - set(hw.columns))}")

    positions = group.sort_values("row").index.to_numpy(dtype=int)
    hidden = work_rows.loc[positions, "row"].to_numpy(dtype=int)
    if len(hidden) == 0 or (hidden < 0).any() or (hidden >= len(hw)).any():
        raise RuntimeError(f"invalid hidden indices for {wid}")
    if len(hidden) > 1 and not np.array_equal(hidden[1:], hidden[:-1] + 1):
        raise RuntimeError(f"hidden suffix is not contiguous for {wid}")

    tvt_input = pd.to_numeric(hw["TVT_input"], errors="coerce").to_numpy(dtype="float64")
    known = np.flatnonzero(np.isfinite(tvt_input))
    if len(known) == 0 or int(hidden[0]) != int(known[-1]) + 1:
        raise RuntimeError(f"hidden suffix does not follow visible prefix for {wid}")
    last = int(known[-1])

    md = pd.to_numeric(hw["MD"], errors="coerce").to_numpy(dtype="float64")
    z = pd.to_numeric(hw["Z"], errors="coerce").to_numpy(dtype="float64")
    md_since = md[hidden] - md[last]
    if not np.isfinite(md_since).all() or (md_since <= 0.0).any():
        raise RuntimeError(f"invalid measured-depth horizon for {wid}")

    pred_before = final_tvt[positions].copy()
    last_visible_u = float(tvt_input[last] + z[last])
    boundary_gap_before = float(pred_before[0] + z[hidden[0]] - last_visible_u)
    move = -np.clip(boundary_gap_before, -CAP_FT, CAP_FT) * np.exp(-md_since / TAU_MD_FT)
    pred_after = pred_before + move
    if not np.isfinite(pred_after).all() or float(np.max(np.abs(move))) > CAP_FT + 1e-10:
        raise RuntimeError(f"invalid continuity move for {wid}")
    final_tvt[positions] = pred_after

    boundary_gap_after = float(pred_after[0] + z[hidden[0]] - last_visible_u)
    report_rows.append({
        "well": str(wid),
        "rows": int(len(hidden)),
        "changed_rows": int(np.count_nonzero(move)),
        "last_visible_row": last,
        "first_hidden_row": int(hidden[0]),
        "boundary_gap_u_before": boundary_gap_before,
        "boundary_gap_u_after": boundary_gap_after,
        "cap": CAP_FT,
        "tau": TAU_MD_FT,
        "first_move": float(move[0]),
        "mean_abs_move": float(np.mean(np.abs(move))),
        "max_abs_move": float(np.max(np.abs(move))),
    })
    contexts[str(wid)] = {
        "md": md,
        "z": z,
        "tvt_input": tvt_input,
        "known": known,
        "hidden": hidden,
        "md_since": md_since,
        "pred_before": pred_before,
        "pred_after": pred_after,
        "move": move,
        "last": last,
    }

report = pd.DataFrame(report_rows)
delta = final_tvt - base_tvt
if not np.isfinite(final_tvt).all() or int(np.count_nonzero(delta)) != EXPECTED_IN_MEMORY_CHANGED_ROWS:
    raise RuntimeError("unexpected final prediction vector or changed-row count")
if prediction_sha(final_tvt) != FINAL_IN_MEMORY_PRED_SHA:
    raise RuntimeError(f"in-memory prediction hash mismatch: {prediction_sha(final_tvt)}")

submission = pd.DataFrame({"id": base["id"].astype(str).to_numpy(), "tvt": final_tvt})
named_path = WORK / "submission_u_continuity_cap8_tau240.csv"
submission_path = WORK / "submission.csv"
submission.to_csv(named_path, index=False, columns=["id", "tvt"], lineterminator="\n")
submission.to_csv(submission_path, index=False, columns=["id", "tvt"], lineterminator="\n")
report.to_csv(WORK / "continuity_refinement_report.csv", index=False, lineterminator="\n")

if file_sha(submission_path) != FINAL_SHA or file_sha(named_path) != FINAL_SHA:
    raise RuntimeError(f"continuity CSV hash mismatch: {file_sha(submission_path)}")
if file_sha(public_final_path) != FINAL_SHA:
    raise RuntimeError(f"public continuity reference changed: {file_sha(public_final_path)}")
reloaded = pd.read_csv(submission_path, dtype={"id": "string"})
if prediction_sha(reloaded["tvt"].to_numpy(dtype="float64")) != FINAL_RELOADED_PRED_SHA:
    raise RuntimeError("reloaded prediction hash mismatch")
if not reloaded["id"].equals(sample["id"]) or list(reloaded.columns) != ["id", "tvt"]:
    raise RuntimeError("final submission violates official id/schema contract")
reloaded_delta = reloaded["tvt"].to_numpy(dtype="float64") - base_tvt
if int(np.count_nonzero(reloaded_delta)) != EXPECTED_RELOADED_CHANGED_ROWS:
    raise RuntimeError("unexpected serialized changed-row count")

audit = {
    "rows": int(len(submission)),
    "columns": list(submission.columns),
    "unique_ids": bool(submission["id"].is_unique),
    "finite_tvt": bool(np.isfinite(final_tvt).all()),
    "official_id_order": True,
    "base_sha256": file_sha(base_path),
    "public_reference_sha256": file_sha(public_final_path),
    "submission_sha256": file_sha(submission_path),
    "in_memory_prediction_sha256": prediction_sha(final_tvt),
    "reloaded_prediction_sha256": prediction_sha(reloaded["tvt"].to_numpy(dtype="float64")),
    "cap_ft": CAP_FT,
    "tau_md_ft": TAU_MD_FT,
    "wells": int(len(report)),
    "in_memory_changed_rows": int(np.count_nonzero(delta)),
    "reloaded_changed_rows": int(np.count_nonzero(reloaded_delta)),
    "reloaded_untouched_rows": int(np.count_nonzero(reloaded_delta == 0.0)),
    "max_abs_move_ft": float(np.max(np.abs(reloaded_delta))),
    "mean_abs_move_ft": float(np.mean(np.abs(reloaded_delta))),
    "score_status": "UNSUBMITTED_NOTEBOOK_RUN",
}
with open(WORK / "u_continuity_artifact_audit.json", "w", encoding="utf-8") as f:
    json.dump(audit, f, indent=2, sort_keys=True)
print(report.to_string(index=False))
print(json.dumps(audit, indent=2, sort_keys=True))
print("U-continuity artifact-locked candidate READY:", file_sha(submission_path))


## VISUALS — geometry, intervention, and integrity

The figures make the transformation inspectable. The first two show why the correction exists and how it fades; the final two show its effect on the submitted trajectories and the complete contract/evidence dashboard. The submission hash is checked again after every figure is saved.


In [ ]:
before_visual_sha = file_sha(submission_path)
colors = {"000d7d20": "#0ea5e9", "00bbac68": "#8b5cf6", "00e12e8b": "#f59e0b"}

fig, axes = plt.subplots(len(EXPECTED_WELLS), 1, figsize=(15, 13), sharex=False)
for ax, wid in zip(axes, EXPECTED_WELLS):
    ctx = contexts[wid]
    known = ctx["known"]
    hidden = ctx["hidden"]
    last = ctx["last"]
    visible = known[max(0, len(known) - 180):]
    keep_hidden = ctx["md_since"] <= 650.0
    x_visible = ctx["md"][visible] - ctx["md"][last]
    u_visible = ctx["tvt_input"][visible] + ctx["z"][visible]
    x_hidden = ctx["md_since"][keep_hidden]
    u_before = ctx["pred_before"][keep_hidden] + ctx["z"][hidden[keep_hidden]]
    u_after = ctx["pred_after"][keep_hidden] + ctx["z"][hidden[keep_hidden]]
    ax.plot(x_visible, u_visible, color="#334155", lw=2.1, label="visible U = TVT_input + Z")
    ax.plot(x_hidden, u_before, color="#ef4444", lw=2.0, ls="--", label="control hidden U")
    ax.plot(x_hidden, u_after, color=colors[wid], lw=2.6, label="continuity-refined hidden U")
    ax.axvline(0.0, color="#0f172a", lw=1.2, alpha=0.55)
    row = report.loc[report["well"].eq(wid)].iloc[0]
    ax.set_title(f"Well {wid}: boundary gap {row.boundary_gap_u_before:+.4f} → {row.boundary_gap_u_after:+.4f} ft")
    ax.set_ylabel("Stratigraphic U (ft)")
    ax.legend(loc="best", frameon=True, ncol=3)
axes[-1].set_xlabel("Measured depth from last visible row (ft)")
fig.suptitle("Visible-to-hidden stratigraphic continuity at the prediction boundary", fontsize=21, fontweight="bold", y=1.003)
fig.tight_layout()
fig.savefig(WORK / "visual_01_boundary_continuity.png", bbox_inches="tight")
plt.show()

fig, ax = plt.subplots(figsize=(14, 7))
for wid in EXPECTED_WELLS:
    ctx = contexts[wid]
    first = float(ctx["move"][0])
    ax.plot(ctx["md_since"], np.maximum(np.abs(ctx["move"]), 1e-14), lw=2.5, color=colors[wid], label=f"{wid} (first {first:+.4f} ft)")
ax.axvline(TAU_MD_FT, color="#0f172a", lw=1.4, ls="--", alpha=0.65, label="one decay constant (240 ft)")
ax.set_xlim(left=0.0)
ax.set_yscale("log")
ax.set_xlabel("Measured depth into hidden suffix (ft)")
ax.set_ylabel("Absolute TVT correction (ft, log scale)")
ax.set_title("Cap-8 / tau-240 intervention: every boundary correction decays smoothly")
ax.legend(title="Well", frameon=True)
fig.tight_layout()
fig.savefig(WORK / "visual_02_exponential_fade.png", bbox_inches="tight")
plt.show()


In [ ]:
fig, axes = plt.subplots(len(EXPECTED_WELLS), 1, figsize=(15, 13), sharex=False)
for ax, wid in zip(axes, EXPECTED_WELLS):
    ctx = contexts[wid]
    hidden = ctx["hidden"]
    ax.plot(hidden, ctx["pred_before"], color="#94a3b8", lw=2.0, label="exact 6.594 control")
    ax.plot(hidden, ctx["pred_after"], color=colors[wid], lw=2.4, label="U-continuity candidate")
    ax.fill_between(hidden, ctx["pred_before"], ctx["pred_after"], color=colors[wid], alpha=0.22)
    ax.set_title(f"Well {wid} — {len(hidden):,} submission rows")
    ax.set_ylabel("Predicted TVT (ft)")
    ax.invert_yaxis()
    ax.legend(loc="best", frameon=True)
axes[-1].set_xlabel("Horizontal-well row index")
fig.suptitle("Submitted trajectory atlas: exact control versus continuity refinement", fontsize=21, fontweight="bold", y=1.003)
fig.tight_layout()
fig.savefig(WORK / "visual_03_trajectory_atlas.png", bbox_inches="tight")
plt.show()

fig = plt.figure(figsize=(16, 8))
gs = fig.add_gridspec(1, 2, width_ratios=[1.05, 1.55])
ax1 = fig.add_subplot(gs[0, 0])
x = np.arange(len(report))
before = np.abs(report["boundary_gap_u_before"].to_numpy(dtype=float))
after = np.abs(report["boundary_gap_u_after"].to_numpy(dtype=float))
w = 0.36
ax1.bar(x - w/2, before, width=w, color="#ef4444", label="before")
ax1.bar(x + w/2, after, width=w, color="#10b981", label="after")
ax1.set_xticks(x, report["well"], rotation=18)
ax1.set_yscale("log")
ax1.set_ylabel("Absolute U boundary gap (ft, log scale)")
ax1.set_title("Boundary discontinuity removed")
ax1.legend(frameon=True)

ax2 = fig.add_subplot(gs[0, 1])
ax2.axis("off")
dashboard = [
    ["Rows / schema", f"{len(submission):,} / id,tvt"],
    ["Changed / untouched", f"{int(np.count_nonzero(reloaded_delta)):,} / {int(np.count_nonzero(reloaded_delta == 0.0)):,}"],
    ["Cap / decay", f"{CAP_FT:.0f} ft / {TAU_MD_FT:.0f} MD-ft"],
    ["Selected validation", "495 wells; pooled ΔRMSE -0.02576"],
    ["Independent re-audit", "773 wells; all 12 routes improved"],
    ["Control SHA-256", BASE_SHA[:18] + "…"],
    ["Final SHA-256", FINAL_SHA[:18] + "…"],
    ["Score state", "UNSUBMITTED — no score claimed"],
]
table = ax2.table(cellText=dashboard, colLabels=["Audit item", "Verified value"], loc="center", cellLoc="left")
table.auto_set_font_size(False)
table.set_fontsize(12)
table.scale(1.0, 2.15)
for (r, c), cell in table.get_celld().items():
    cell.set_edgecolor("white")
    cell.set_facecolor("#0f172a" if r == 0 else ("#e0f2fe" if r % 2 else "#f8fafc"))
    cell.set_text_props(color="white" if r == 0 else "#0f172a", weight="bold" if r == 0 else "normal")
ax2.set_title("Evidence and submission-integrity dashboard", fontsize=18, fontweight="bold", pad=20)
fig.suptitle("ROGII U-continuity frontier — transparent evidence, exact artifact", fontsize=22, fontweight="bold")
fig.tight_layout()
fig.savefig(WORK / "visual_04_evidence_integrity_dashboard.png", bbox_inches="tight")
plt.show()

after_visual_sha = file_sha(submission_path)
visuals = [
    "visual_01_boundary_continuity.png",
    "visual_02_exponential_fade.png",
    "visual_03_trajectory_atlas.png",
    "visual_04_evidence_integrity_dashboard.png",
]
visual_manifest = {
    "visualizations": visuals,
    "all_nonempty": all((WORK / name).exists() and (WORK / name).stat().st_size > 0 for name in visuals),
    "submission_sha256_before": before_visual_sha,
    "submission_sha256_after": after_visual_sha,
    "submission_unchanged": before_visual_sha == after_visual_sha == FINAL_SHA,
    "score_status": "UNSUBMITTED_NOTEBOOK_RUN",
}
with open(WORK / "visualization_manifest.json", "w", encoding="utf-8") as f:
    json.dump(visual_manifest, f, indent=2, sort_keys=True)
if not visual_manifest["all_nonempty"] or not visual_manifest["submission_unchanged"]:
    raise RuntimeError("visualization integrity audit failed")
print(json.dumps(visual_manifest, indent=2, sort_keys=True))
print("U-continuity visualization audit PASS")


## Experimental bounded C1 continuity

The exact audited C0 artifact is the immutable input. This layer matches local slope only, keeps the first hidden prediction unchanged, fades to zero, and caps peak movement at 1 ft. This is a deployment diagnostic, not evidence of leaderboard improvement.

In [ ]:
# C1_BOUNDED_SLOPE_FORK: one preregistered target-free slope correction.
C1_BASE_SHA = FINAL_SHA
C1_WINDOW_MD_FT = 240.0
C1_TAU_MD_FT = 120.0
C1_MAX_MOVE_FT = 1.0

if file_sha(submission_path) != C1_BASE_SHA:
    raise RuntimeError(f"C1 expected exact audited C0 base {C1_BASE_SHA}, got {file_sha(submission_path)}")

c1_base = pd.read_csv(submission_path, dtype={"id": "string"})
c1_values = c1_base["tvt"].to_numpy(dtype="float64").copy()
c1_report_rows = []


def weighted_local_slope(x, y, weights):
    x_center = np.average(x, weights=weights)
    y_center = np.average(y, weights=weights)
    denominator = np.sum(weights * (x - x_center) ** 2)
    if denominator <= 0:
        raise RuntimeError("degenerate C1 slope window")
    return float(np.sum(weights * (x - x_center) * (y - y_center)) / denominator)


for wid, group in work_rows.groupby("well", sort=False):
    ctx = contexts[str(wid)]
    positions = group.sort_values("row").index.to_numpy(dtype=int)
    known, hidden, last = ctx["known"], ctx["hidden"], ctx["last"]
    md, z, tvt_input = ctx["md"], ctx["z"], ctx["tvt_input"]

    visible_distance = md[known] - md[last]
    visible_mask = visible_distance >= -C1_WINDOW_MD_FT
    hidden_distance = md[hidden] - md[hidden[0]]
    hidden_mask = hidden_distance <= C1_WINDOW_MD_FT
    if int(visible_mask.sum()) < 30 or int(hidden_mask.sum()) < 30:
        raise RuntimeError(f"insufficient C1 slope window for {wid}")

    visible_x = visible_distance[visible_mask]
    visible_u = (tvt_input[known] + z[known])[visible_mask]
    hidden_x = hidden_distance[hidden_mask]
    hidden_u = (c1_values[positions] + z[hidden])[hidden_mask]
    visible_slope = weighted_local_slope(
        visible_x, visible_u, np.exp(-np.abs(visible_x) / C1_TAU_MD_FT)
    )
    hidden_slope = weighted_local_slope(
        hidden_x, hidden_u, np.exp(-hidden_x / C1_TAU_MD_FT)
    )
    raw_slope_gap = visible_slope - hidden_slope
    slope_cap = C1_MAX_MOVE_FT * np.e / C1_TAU_MD_FT
    applied_slope_gap = float(np.clip(raw_slope_gap, -slope_cap, slope_cap))
    move = applied_slope_gap * hidden_distance * np.exp(-hidden_distance / C1_TAU_MD_FT)
    if abs(float(move[0])) > 1e-12 or float(np.max(np.abs(move))) > C1_MAX_MOVE_FT + 1e-10:
        raise RuntimeError(f"invalid C1 move for {wid}")
    c1_values[positions] += move
    c1_report_rows.append({
        "well": str(wid), "rows": int(len(hidden)),
        "visible_slope": visible_slope, "hidden_slope": hidden_slope,
        "raw_slope_gap": raw_slope_gap, "applied_slope_gap": applied_slope_gap,
        "clipped": bool(abs(raw_slope_gap - applied_slope_gap) > 1e-12),
        "first_move": float(move[0]), "mean_abs_move": float(np.mean(np.abs(move))),
        "max_abs_move": float(np.max(np.abs(move))),
    })

c1_submission = pd.DataFrame({"id": c1_base["id"], "tvt": c1_values})
c1_named_path = WORK / "submission_u_continuity_c0_c1_cap1_tau120.csv"
c1_submission.to_csv(c1_named_path, index=False, columns=["id", "tvt"], lineterminator="\n")
c1_submission.to_csv(submission_path, index=False, columns=["id", "tvt"], lineterminator="\n")
c1_report = pd.DataFrame(c1_report_rows)
c1_report.to_csv(WORK / "c1_continuity_report.csv", index=False, lineterminator="\n")

c1_reloaded = pd.read_csv(submission_path, dtype={"id": "string"})
if list(c1_reloaded.columns) != ["id", "tvt"] or not c1_reloaded["id"].equals(sample["id"]):
    raise RuntimeError("C1 submission violates official schema/order")
if not np.isfinite(c1_reloaded["tvt"].to_numpy(dtype="float64")).all():
    raise RuntimeError("C1 submission contains non-finite predictions")
c1_delta = c1_reloaded["tvt"].to_numpy(dtype="float64") - c1_base["tvt"].to_numpy(dtype="float64")
if float(np.max(np.abs(c1_delta))) > C1_MAX_MOVE_FT + 1e-9:
    raise RuntimeError("serialized C1 correction exceeds cap")

c1_audit = {
    "method": "bounded C1 U-continuity",
    "score_status": "UNSUBMITTED_NOTEBOOK_RUN",
    "rows": int(len(c1_reloaded)), "wells": int(len(c1_report)),
    "base_c0_sha256": C1_BASE_SHA, "final_c1_sha256": file_sha(submission_path),
    "window_md_ft": C1_WINDOW_MD_FT, "tau_md_ft": C1_TAU_MD_FT,
    "max_move_cap_ft": C1_MAX_MOVE_FT,
    "changed_rows": int(np.count_nonzero(c1_delta)),
    "mean_abs_move_ft": float(np.mean(np.abs(c1_delta))),
    "max_abs_move_ft": float(np.max(np.abs(c1_delta))),
    "clipped_wells": int(c1_report["clipped"].sum()),
    "first_hidden_points_preserved": bool((c1_report["first_move"].abs() <= 1e-12).all()),
}
with open(WORK / "c1_continuity_audit.json", "w", encoding="utf-8") as f:
    json.dump(c1_audit, f, indent=2, sort_keys=True)

fig, ax = plt.subplots(figsize=(14, 6))
for wid, group in work_rows.groupby("well", sort=False):
    positions = group.sort_values("row").index.to_numpy(dtype=int)
    hidden = contexts[str(wid)]["hidden"]
    distance = contexts[str(wid)]["md"][hidden] - contexts[str(wid)]["md"][hidden[0]]
    ax.plot(distance, c1_delta[positions], lw=2.2, label=str(wid))
ax.axhline(0.0, color="#0f172a", lw=1.0)
ax.set(xlabel="Measured depth from first hidden row (ft)", ylabel="C1 TVT correction (ft)",
       title="Bounded C1 slope correction: zero at boundary, exponential fade")
ax.legend(frameon=True)
fig.tight_layout()
fig.savefig(WORK / "visual_05_c1_slope_correction.png", bbox_inches="tight")
plt.show()
if file_sha(submission_path) != c1_audit["final_c1_sha256"]:
    raise RuntimeError("C1 visualization changed submission")

print(c1_report.to_string(index=False))
print(json.dumps(c1_audit, indent=2, sort_keys=True))
print("C1 candidate READY:", file_sha(submission_path))


## Final output contract

`submission.csv` is the bounded C0+C1 experimental candidate. `c1_continuity_report.csv` and `c1_continuity_audit.json` record every slope and movement. Score remains unclaimed until Kaggle submission reaches `COMPLETE`.